# Ejercicio 9: API de Google Gemini - Versión para Jupyter Local

Este notebook está diseñado para ejecutarse en tu máquina local con Jupyter.

## ⚠️ IMPORTANTE - LEE PRIMERO:

Antes de ejecutar nada:
1. Obtén tu API Key en: https://makersuite.google.com/app/apikey
2. Ten la clave lista para copiarla en la siguiente celda
3. Ejecuta las celdas **en orden** sin saltarte ninguna

## PASO 1: Instalar dependencias

Ejecuta esta celda una sola vez. Tardará unos minutos.

In [ ]:
import subprocess
import sys

print("📦 Instalando dependencias...")
print("Esto puede tomar 2-3 minutos\n")

packages = [
    'google-generativeai',
    'scikit-learn',
    'sentence-transformers',
    'python-dotenv',
    'numpy',
    'pandas'
]

for package in packages:
    print(f"Instalando {package}...", end=" ")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print("✓")

print("\n✓ ¡Instalación completada!")

## PASO 2: Guardar tu API Key

En la siguiente celda, reemplaza `"tu_api_key_aqui"` con tu clave real.

**Ejemplo:**
```python
API_KEY = "AIzaSyDxxxxxxxxxxx"  # Tu verdadera clave
```

In [ ]:
# ⭐ EDITA ESTA LÍNEA CON TU API KEY
API_KEY = "tu_api_key_aqui"

# Validar que se haya configurado
if API_KEY == "tu_api_key_aqui":
    print("❌ ERROR: No has configurado tu API Key")
    print("\nPor favor:")
    print("1. Ve a https://makersuite.google.com/app/apikey")
    print("2. Copia tu clave")
    print("3. Reemplaza 'tu_api_key_aqui' en la línea anterior")
    print("4. Vuelve a ejecutar esta celda")
else:
    print(f"✓ API Key configurada (primeros 10 caracteres: {API_KEY[:10]}...)")

## PASO 3: Importar librerías y configurar

In [ ]:
import google.generativeai as genai
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("📚 Importando librerías...")

# Configurar Gemini
genai.configure(api_key=API_KEY)

print("✓ Librerías cargadas")
print("✓ API configurada")

## PASO 4: Probar conexión a Google Gemini

In [ ]:
print("🔗 Probando conexión con Google Gemini...\n")

try:
    # Listar modelos disponibles
    print("Modelos disponibles:")
    models = genai.list_models()
    for model in models:
        print(f"  ✓ {model.display_name}")
    
    print("\n" + "="*60)
    # Hacer prueba simple
    model = genai.GenerativeModel('gemini-pro')
    response = model.generate_content("¿Cuál es el resultado de 15 + 27? Responde solo con el número.")
    
    print(f"Prueba: ¿Cuál es 15 + 27?")
    print(f"Respuesta: {response.text}")
    print("="*60)
    
    print("\n✓ ¡Conexión EXITOSA! Todo está funcionando correctamente")
    
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("\nPosibles soluciones:")
    print("1. Verifica que tu API Key sea correcta")
    print("2. Asegúrate de tener conexión a Internet")
    print("3. Prueba obtener una nueva clave en: https://makersuite.google.com/app/apikey")

## PASO 5: Cargar el dataset 20 News Groups

In [ ]:
print("📥 Descargando dataset 20 News Groups...")
print("(Primera vez tardará más, luego se cachea)\n")

try:
    # Descargar dataset
    newsgroups = fetch_20newsgroups(
        subset='train',
        categories=['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med'],
        remove=('headers', 'footers', 'quotes')
    )
    
    # Tomar solo los primeros 100 documentos
    documents = newsgroups.data[:100]
    
    print(f"✓ Dataset descargado exitosamente")
    print(f"\n📊 Información:")
    print(f"  - Total de documentos: {len(documents)}")
    print(f"  - Categorías: {', '.join(newsgroups.target_names)}")
    print(f"\n📄 Ejemplo (primeros 300 caracteres del primer documento):")
    print(f"{'-'*70}")
    print(documents[0][:300] + "...")
    print(f"{'-'*70}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Intenta ejecutar nuevamente la celda")

## PASO 6: Generar embeddings de los documentos

Los embeddings son representaciones numéricas de los documentos que permiten comparar similitud.

In [ ]:
print("🤖 Cargando modelo de embeddings...")
print("(Este es un modelo pequeño y rápido)\n")

try:
    embedding_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
    print("✓ Modelo cargado\n")
    
    print("⏳ Generando embeddings para los documentos...")
    print("(Esto tarda ~30-60 segundos)\n")
    
    embeddings = embedding_model.encode(documents, show_progress_bar=True)
    
    print(f"\n✓ Embeddings generados exitosamente")
    print(f"\n📐 Información técnica:")
    print(f"  - Número de documentos: {embeddings.shape[0]}")
    print(f"  - Dimensión de cada embedding: {embeddings.shape[1]}")
    print(f"  - Tamaño en memoria: {embeddings.nbytes / 1024 / 1024:.2f} MB")
    
except Exception as e:
    print(f"❌ Error: {e}")

## PASO 7: Búsqueda de documentos similares

In [ ]:
# Crear una query de prueba
query = "¿Cuáles son los tratamientos para enfermedades del corazón?"

print(f"🔍 QUERY: {query}")
print(f"{'-'*80}\n")

try:
    # Generar embedding para la query
    query_embedding = embedding_model.encode([query])
    
    # Calcular similaridad
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Obtener los 5 más similares
    top_5_indices = np.argsort(similarities)[-5:][::-1]
    
    print("TOP 5 DOCUMENTOS MÁS SIMILARES:")
    print("="*80)
    
    for i, idx in enumerate(top_5_indices, 1):
        similarity_score = similarities[idx]
        document = documents[idx]
        
        print(f"\n{i}. Documento #{idx}")
        print(f"   Similaridad: {similarity_score:.4f} (escala 0-1)")
        print(f"{'-'*80}")
        print(document[:400])
        if len(document) > 400:
            print("   ...")
    
    print(f"\n\n✓ Búsqueda completada")
        
except Exception as e:
    print(f"❌ Error: {e}")

## PASO 8: Implementar RAG (Retrieval-Augmented Generation)

Esto combina búsqueda de documentos + generación de respuestas con Gemini

In [ ]:
def rag_query(query_text, top_k=3):
    """
    Función RAG:
    1. Recupera documentos similares a la query
    2. Los usa como contexto
    3. Gemini genera una respuesta basada en ese contexto
    """
    try:
        # Generar embedding de la query
        query_emb = embedding_model.encode([query_text])
        
        # Calcular similaridad
        sims = cosine_similarity(query_emb, embeddings)[0]
        
        # Obtener top K documentos
        top_indices = np.argsort(sims)[-top_k:][::-1]
        
        # Construir contexto
        context_parts = []
        for i, idx in enumerate(top_indices, 1):
            context_parts.append(f"[Documento {i}]\n{documents[idx][:350]}")
        
        context = "\n---\n".join(context_parts)
        
        # Crear prompt para Gemini
        prompt = f"""Basándote ÚNICAMENTE en los siguientes documentos, responde la pregunta.

[CONTEXTO]
{context}
[FIN DEL CONTEXTO]

Pregunta: {query_text}

Instrucciones:
- Responde solo con información de los documentos
- Si no hay información relevante, dilo claramente
- Sé conciso

Respuesta:"""
        
        # Generar respuesta
        model = genai.GenerativeModel('gemini-pro')
        response = model.generate_content(prompt)
        
        return {
            'query': query_text,
            'top_indices': top_indices,
            'similarities': [sims[idx] for idx in top_indices],
            'response': response.text,
            'success': True
        }
    except Exception as e:
        return {
            'query': query_text,
            'response': f'Error: {str(e)}',
            'success': False
        }

print("✓ Función RAG definida correctamente")

## PASO 9: Ejemplo 1 de RAG

In [ ]:
query1 = "¿Cuáles son los tópicos principales de los documentos?"

print(f"📝 QUERY: {query1}")
print(f"{'='*80}\n")
print("⏳ Procesando...\n")

result = rag_query(query1, top_k=3)

if result['success']:
    print(f"📊 Documentos recuperados:")
    for i, (idx, sim) in enumerate(zip(result['top_indices'], result['similarities']), 1):
        print(f"  {i}. Documento #{idx} (similaridad: {sim:.4f})")
    
    print(f"\n{'='*80}")
    print(f"💬 RESPUESTA DE GEMINI:")
    print(f"{'='*80}")
    print(result['response'])
else:
    print(f"❌ Error: {result['response']}")

## PASO 10: Ejemplo 2 de RAG

In [ ]:
query2 = "¿Hay información sobre creencias religiosas?"

print(f"📝 QUERY: {query2}")
print(f"{'='*80}\n")
print("⏳ Procesando...\n")

result = rag_query(query2, top_k=2)

if result['success']:
    print(f"📊 Documentos recuperados:")
    for i, (idx, sim) in enumerate(zip(result['top_indices'], result['similarities']), 1):
        print(f"  {i}. Documento #{idx} (similaridad: {sim:.4f})")
    
    print(f"\n{'='*80}")
    print(f"💬 RESPUESTA DE GEMINI:")
    print(f"{'='*80}")
    print(result['response'])
else:
    print(f"❌ Error: {result['response']}")

## PASO 11: Ejemplo 3 de RAG

In [ ]:
query3 = "¿Qué se habla sobre gráficos computacionales?"

print(f"📝 QUERY: {query3}")
print(f"{'='*80}\n")
print("⏳ Procesando...\n")

result = rag_query(query3, top_k=2)

if result['success']:
    print(f"📊 Documentos recuperados:")
    for i, (idx, sim) in enumerate(zip(result['top_indices'], result['similarities']), 1):
        print(f"  {i}. Documento #{idx} (similaridad: {sim:.4f})")
    
    print(f"\n{'='*80}")
    print(f"💬 RESPUESTA DE GEMINI:")
    print(f"{'='*80}")
    print(result['response'])
else:
    print(f"❌ Error: {result['response']}")

## PASO 12: ¡Prueba con TUS propias queries!

Edita la variable `mi_query` abajo con tu propia pregunta y ejecuta la celda

In [ ]:
# ⭐ EDITA ESTA PREGUNTA CON LA TUYA
mi_query = "¿Cuáles son los temas principales de los documentos?"

print(f"📝 TU QUERY: {mi_query}")
print(f"{'='*80}\n")
print("⏳ Buscando documentos relevantes y generando respuesta...\n")

result = rag_query(mi_query, top_k=3)

if result['success']:
    print(f"📊 DOCUMENTOS RECUPERADOS:")
    for i, (idx, sim) in enumerate(zip(result['top_indices'], result['similarities']), 1):
        print(f"\n  {i}. Documento #{idx}")
        print(f"     Similaridad: {sim:.4f} ({int(sim*100)}% de similitud)")
    
    print(f"\n\n{'='*80}")
    print(f"💬 RESPUESTA GENERADA POR GEMINI:")
    print(f"{'='*80}")
    print(result['response'])
else:
    print(f"❌ Error: {result['response']}")

---

## 📚 Resumen de lo aprendido:

### Conceptos clave:

| Concepto | Explicación |
|----------|-------------|
| **API de Gemini** | Acceso a modelos de IA generativa de Google |
| **Embeddings** | Transformación de texto en vectores numéricos |
| **Similitud del Coseno** | Método para medir cuán similares son dos textos |
| **RAG** | Combina búsqueda + generación de texto (Retrieval-Augmented Generation) |

### Casos de uso reales:
- 💬 Chatbots inteligentes
- 🔎 Búsqueda semántica
- 📄 Asistentes de documentos
- ❓ Sistemas de preguntas y respuestas
- 📚 Análisis de textos

### Flujo de RAG:
```
Usuario pregunta
       ↓
Buscar documentos similares
       ↓
Usar esos documentos como contexto
       ↓
Gemini genera una respuesta
       ↓
Usuario obtiene respuesta precisa
```

---

¡Felicidades! 🎉 Has aprendido cómo usar Google Gemini con búsqueda semántica y generación de texto.